# Car image classifier
Upload an image and click **Classify**. The model is downloaded from a Hugging Face model repository when the app starts.

In [ ]:
from pathlib import Path
from fastai.vision.all import load_learner, PILImage
import ipywidgets as widgets
from IPython.display import display
from huggingface_hub import hf_hub_download

# CHANGE THIS to your Hugging Face model repository, e.g. "yourname/car-classifier"
MODEL_REPO = "hariiir/modelt"
MODEL_FILENAME = "exportv2.pkl"

if MODEL_REPO.startswith("YOUR_"):
    raise ValueError("Set MODEL_REPO to your public Hugging Face model repository before deploying.")

model_path = hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILENAME)
learn_inf = load_learner(model_path)

In [ ]:
btn_upload = widgets.FileUpload(accept="image/*", multiple=False)
btn_run = widgets.Button(description="Classify", button_style="primary")
out_pl = widgets.Output()
lbl_pred = widgets.Label(value="Upload a car image to begin.")

def on_click_classify(_):
    if not btn_upload.value:
        lbl_pred.value = "Please upload an image first."
        return
    try:
        upload = btn_upload.value
        item = next(iter(upload.values())) if isinstance(upload, dict) else upload[0]
        content = item["content"]
        img = PILImage.create(bytes(content))
        with out_pl:
            out_pl.clear_output(wait=True)
            display(img.to_thumb(256, 256))
        pred, pred_idx, probs = learn_inf.predict(img)
        lbl_pred.value = f"Prediction: {pred}; Probability: {float(probs[pred_idx]):.4f}"
    except Exception as exc:
        lbl_pred.value = f"Error: {exc}"

btn_run.on_click(on_click_classify)
display(widgets.VBox([widgets.HTML("<h3>Car image classifier</h3>"), btn_upload, btn_run, out_pl, lbl_pred]))